# Ders 10 - Üretimde Yapay Zeka Ajanları

Bu derste, Microsoft Agent Framework (Python) kullanarak yapay zeka ajanları için **üretim desenlerini** öğreneceksiniz. Konular şunlardır:

- **Gözlemlenebilirlik** — ajan etkileşimlerine zamanlama ve günlük kaydı ekleme
- **Değerlendirme** — yanıt kalitesini puanlamak için bir değerlendirme ajanı kullanma
- **Maliyet yönetimi** — token optimizasyonu ve model seçimi stratejileri

Senaryo, kullanıcıların seyahat planlamasına yardımcı olan bir **seyahat acentesi**, üzerine izleme ve değerlendirme katmanları eklenmiş olarak.


## Kurulum


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Üretim Dikkat Edilmesi Gerekenler

AI ajanlarını prototiplerden üretime taşımak, üç temel ayağa dikkat gerektirir:

1. **Gözlemlenebilirlik** — Ajanın ne yaptığını, ne kadar sürdüğünü ve hangi araçları çağırdığını görmeniz gerekir. İzleme ve kayıt olmadan, üretimdeki sorunları hata ayıklamak neredeyse imkansızdır.

2. **Değerlendirme** — Otomatik kalite kontrolleri, ajan cevaplarının zamanla doğru, eksiksiz ve faydalı kalmasını sağlar. Bir değerlendirme ajanı, cevapları tanımlanmış kriterlere göre puanlayabilir.

3. **Maliyet Yönetimi** — Token kullanımı doğrudan maliyeti etkiler. Prompt optimizasyonu, model seçimi ve önbellekleme gibi stratejiler, kaliteyi düşürmeden giderlerin kontrol altında kalmasına yardımcı olur.


## Gözlemlenebilir Bir Ajan Oluşturma

Seyahat araçlarını tanımlıyoruz ve gecikmeyi izleyebilmek için ajan çağrısını zamanlama ile sarıyoruz. Üretimde OpenTelemetry veya benzeri bir izleme altyapısı ile entegre olursunuz.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Değerlendirme Kalıpları

Yaygın bir üretim kalıbı, ikinci bir temsilciyi **değerlendirici** olarak kullanmaktır. Değerlendirici, birincil temsilcinin yanıtını tamlık, doğruluk ve faydalılık gibi önceden tanımlanmış kriterlere göre puanlar.

Bu şunları sağlar:
- Yanıtlar kullanıcılara ulaşmadan önce otomatik kalite kontrolleri
- İstekler veya modeller değiştiğinde regresyon tespiti
- Temsilci performansının zaman içinde sürekli izlenmesi


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Maliyet Yönetimi Stratejileri

Maliyetleri kontrol etmek, üretim AI ajanları için kritik öneme sahiptir. İşte temel stratejiler:

| Strateji | Açıklama |
|---|---|
| **İstemi optimize etme** | Sistem talimatlarını kısa tutun. Girdi tokenlarını azaltmak için gereksiz bağlamı kaldırın. |
| **Model seçimi** | Sınıflandırma veya çıkarım gibi basit görevler için daha küçük, daha ucuz modelleri (örneğin GPT-4o-mini) kullanın ve karmaşık akıl yürütme için daha büyük modelleri ayırın. |
| **Önbelleğe alma** | Araç sonuçlarını ve sık sorguları önbelleğe alınarak gereksiz API çağrılarından kaçının. |
| **Token bütçeleri** | Beklenmedik uzun yanıtları önlemek için `max_tokens` sınırları belirleyin. |
| **Gruplama** | Mümkün olduğunda birden fazla kullanıcı sorgusunu tek bir API çağrısında gruplayın. |

Uygulamada, aşamalı bir yaklaşım iyi çalışır: Basit istekleri hızlı ve uygun maliyetli bir modele yönlendirin ve yalnızca karmaşık sorguları daha yetenekli modele yükseltin.


## Özet

Bu derste şunları öğrendiniz:

1. **Gözlemlenebilirlik eklemek**, zamanlama ve günlük kaydı ile ajan etkileşimlerini izleyerek izleme ve takip için temel oluşturmak.
2. **Ajan yanıtlarını değerlendirmek**, tamamlık, doğruluk ve faydalılık açısından puanlayan bir değerlendirici ajan kullanarak otomatik olarak değerlendirmek.
3. **Maliyetleri yönetmek**, prompt optimizasyonu, model seçimi, önbellekleme ve token bütçeleri aracılığıyla.

Bu üretim desenleri, yapay zeka ajanlarınızın güvenilir, ölçülebilir ve maliyet etkin olmasını sağlar.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
